# BƯỚC 3 – TÍCH HỢP DỮ LIỆU: CONCAT & MERGE

Notebook thực hiện:
1. Kiểm tra và chuẩn hóa `cgv_movie_details.csv` và `cgv_upcoming_movie(s).csv`.
2. Dùng `pd.concat()` để nối dọc thành **Bảng Dữ liệu Tĩnh Hợp nhất**.
3. Kiểm tra khóa `MovieName`.
4. Dùng `pd.merge()` để kết hợp bảng tĩnh với `cgv_showtimes_AI_generated.csv`.
5. Giải thích lý do chọn kiểu gộp.

Đặt 3 file CSV trong cùng thư mục với notebook.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# Thư mục chứa các file CSV
DATA_DIR = Path(".")

def find_file(candidates):
    for name in candidates:
        path = DATA_DIR / name
        if path.exists():
            return path
    raise FileNotFoundError(
        "Không tìm thấy file. Đã kiểm tra: " + ", ".join(candidates)
    )

details_path = find_file(["cgv_movie_details.csv"])
upcoming_path = find_file([
    "cgv_upcoming_movie.csv",
    "cgv_upcoming_movies.csv"
])
showtimes_ai_path = find_file([
    "cgv_showtimes_AI_generated.csv",
    "cgv_showtimes_ai.csv"
])

print("DETAILS :", details_path)
print("UPCOMING:", upcoming_path)
print("SHOWTIMES AI:", showtimes_ai_path)


In [ ]:
# Đọc dữ liệu
df_details = pd.read_csv(details_path, encoding="utf-8-sig")
df_upcoming = pd.read_csv(upcoming_path, encoding="utf-8-sig")
df_showtimes_ai = pd.read_csv(showtimes_ai_path, encoding="utf-8-sig")

print("Kích thước:")
print("cgv_movie_details     :", df_details.shape)
print("cgv_upcoming_movie(s) :", df_upcoming.shape)
print("cgv_showtimes_AI      :", df_showtimes_ai.shape)

display(df_details.head(3))
display(df_upcoming.head(3))
display(df_showtimes_ai.head(3))


## 1. Kiểm tra cấu trúc hai bảng tĩnh

Hai bảng phải có cùng ý nghĩa bản ghi và cùng bộ cột trước khi `pd.concat()`.

Schema mục tiêu:

`MovieName, MovieURL, Rating, Director, Actors, Genre, ReleaseDate, Duration, Language`


In [ ]:
def normalize_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )
    return df

df_details = normalize_column_names(df_details)
df_upcoming = normalize_column_names(df_upcoming)
df_showtimes_ai = normalize_column_names(df_showtimes_ai)

print("Cột DETAILS :", df_details.columns.tolist())
print("Cột UPCOMING:", df_upcoming.columns.tolist())
print("Cột SHOWTIMES AI:", df_showtimes_ai.columns.tolist())

detail_cols = set(df_details.columns)
upcoming_cols = set(df_upcoming.columns)

print("\nCột có ở DETAILS nhưng thiếu ở UPCOMING:",
      detail_cols - upcoming_cols)
print("Cột có ở UPCOMING nhưng thiếu ở DETAILS:",
      upcoming_cols - detail_cols)


## 2. Chuẩn hóa dữ liệu trước `concat()`

Thực hiện:
- Chuẩn hóa tên cột.
- Xóa khoảng trắng thừa ở dữ liệu text.
- Chuẩn hóa `MovieName` để làm khóa.
- Chuyển `ReleaseDate` về kiểu ngày.
- Bổ sung cột còn thiếu.
- Đưa hai bảng về cùng schema.
- Loại bỏ dòng trùng hoàn toàn.


In [ ]:
STATIC_COLUMNS = [
    "MovieName", "MovieURL", "Rating", "Director", "Actors",
    "Genre", "ReleaseDate", "Duration", "Language"
]

# Bổ sung cột thiếu
for col in STATIC_COLUMNS:
    if col not in df_details.columns:
        df_details[col] = pd.NA
    if col not in df_upcoming.columns:
        df_upcoming[col] = pd.NA

# Cùng thứ tự cột
df_details = df_details[STATIC_COLUMNS].copy()
df_upcoming = df_upcoming[STATIC_COLUMNS].copy()

TEXT_COLS = [
    "MovieName", "MovieURL", "Rating", "Director",
    "Actors", "Genre", "Duration", "Language"
]

def clean_text_columns(df):
    df = df.copy()
    for col in TEXT_COLS:
        df[col] = (
            df[col].fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        df[col] = df[col].replace({"": pd.NA})
    return df

df_details = clean_text_columns(df_details)
df_upcoming = clean_text_columns(df_upcoming)

# Chuẩn hóa khóa MovieName
for df in [df_details, df_upcoming]:
    df["MovieName"] = (
        df["MovieName"].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Chuẩn hóa ngày
df_details["ReleaseDate"] = pd.to_datetime(
    df_details["ReleaseDate"], dayfirst=True, errors="coerce"
)
df_upcoming["ReleaseDate"] = pd.to_datetime(
    df_upcoming["ReleaseDate"], dayfirst=True, errors="coerce"
)

print("Schema DETAILS:")
display(df_details.dtypes)

print("Schema UPCOMING:")
display(df_upcoming.dtypes)


In [ ]:
def quality_report(df, name):
    print(f"===== {name} =====")
    print("Số dòng:", len(df))
    print("Số cột:", len(df.columns))
    print("\nMissing:")
    display(df.isna().sum().to_frame("Missing"))
    print("Dòng trùng hoàn toàn:", df.duplicated().sum())
    print("MovieName trùng:", df["MovieName"].duplicated().sum())
    print()

quality_report(df_details, "CGV MOVIE DETAILS")
quality_report(df_upcoming, "CGV UPCOMING MOVIES")

overlap = (
    set(df_details["MovieName"].dropna())
    & set(df_upcoming["MovieName"].dropna())
)

print("Số MovieName xuất hiện ở cả hai nguồn:", len(overlap))
print(list(sorted(overlap))[:20])


## 3. `pd.concat()` – nối dọc

Sau khi hai bảng đã có cùng schema, dùng `pd.concat(axis=0)` để đặt các dòng của phim sắp chiếu xuống dưới các dòng phim chi tiết.

Kết quả là **Bảng Dữ liệu Tĩnh Hợp nhất**.


In [ ]:
df_details_clean = df_details.drop_duplicates().copy()
df_upcoming_clean = df_upcoming.drop_duplicates().copy()

df_static_master = pd.concat(
    [df_details_clean, df_upcoming_clean],
    axis=0,
    ignore_index=True
)

print("===== BẢNG DỮ LIỆU TĨNH HỢP NHẤT =====")
print("Kích thước:", df_static_master.shape)
display(df_static_master.head())
display(df_static_master.tail())


In [ ]:
# Kiểm tra khóa sau concat
duplicate_movies = (
    df_static_master[
        df_static_master["MovieName"].duplicated(keep=False)
    ]
    .sort_values("MovieName")
)

print("Số dòng thuộc MovieName bị trùng:", len(duplicate_movies))

if len(duplicate_movies) > 0:
    display(duplicate_movies.head(20))
    print(
        "Nếu cùng MovieName có nhiều bản ghi, merge theo MovieName "
        "có thể làm nhân bản suất chiếu. Vì vậy sẽ tạo bảng khóa duy nhất "
        "trước khi merge."
    )


## 4. Chuẩn hóa dữ liệu động

`cgv_showtimes_AI_generated.csv` là dữ liệu động, gồm lịch chiếu và dữ liệu văn bản do AI tạo sinh.

Khóa kết nối với bảng tĩnh là `MovieName`.


In [ ]:
if "MovieName" not in df_showtimes_ai.columns:
    raise KeyError(
        "cgv_showtimes_AI_generated.csv phải có cột MovieName để merge."
    )

df_showtimes_ai["MovieName"] = (
    df_showtimes_ai["MovieName"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

if "ShowDate" in df_showtimes_ai.columns:
    df_showtimes_ai["ShowDate"] = pd.to_datetime(
        df_showtimes_ai["ShowDate"], dayfirst=True, errors="coerce"
    )

print("Số dòng dữ liệu động:", len(df_showtimes_ai))
display(df_showtimes_ai.head())


## 5. Chuẩn bị bảng tĩnh cho `merge()`

Ta tạo một bảng tra cứu trong đó mỗi `MovieName` xuất hiện một lần.

Việc này giúp tránh trường hợp một suất chiếu bị nhân thành nhiều dòng do khóa `MovieName` bị trùng ở bảng tĩnh.


In [ ]:
df_static_master_unique = (
    df_static_master
    .drop_duplicates(subset=["MovieName"], keep="first")
    .copy()
)

print("Số dòng bảng tĩnh trước xử lý khóa:",
      len(df_static_master))
print("Số dòng bảng tĩnh sau xử lý khóa:",
      len(df_static_master_unique))
print("Số MovieName duy nhất:",
      df_static_master_unique["MovieName"].nunique())

static_keys = set(df_static_master_unique["MovieName"].dropna())
dynamic_keys = set(df_showtimes_ai["MovieName"].dropna())

not_found = sorted(dynamic_keys - static_keys)

print("\nMovieName trong dữ liệu động nhưng không có ở bảng tĩnh:",
      len(not_found))
if not_found:
    print(not_found[:20])


## 6. `pd.merge()` – gộp ngang

### Kiểu gộp: `LEFT JOIN`

Ta dùng:

```python
pd.merge(
    df_showtimes_ai,
    df_static_master_unique,
    on="MovieName",
    how="left"
)
```

**Lý do chọn `LEFT JOIN`:**

- Dữ liệu động `cgv_showtimes_AI_generated` là bảng chính.
- Cần giữ lại **toàn bộ suất chiếu**.
- Khi `MovieName` khớp, bổ sung thông tin phim từ bảng tĩnh.
- Nếu không khớp, suất chiếu vẫn được giữ và các cột tĩnh có thể là `NaN`.

Không chọn `INNER JOIN` vì các suất chiếu không khớp sẽ bị loại bỏ.

Không chọn `RIGHT JOIN` vì mục tiêu không phải giữ toàn bộ bảng tĩnh.

Không chọn `OUTER JOIN` vì có thể tạo thêm các phim không có suất chiếu trong bảng kết quả.

Vì vậy `LEFT JOIN` phù hợp với mục tiêu: **lấy lịch chiếu làm dữ liệu chính và bổ sung thông tin phim**.


In [ ]:
df_final = pd.merge(
    df_showtimes_ai,
    df_static_master_unique,
    on="MovieName",
    how="left",
    suffixes=("", "_Static"),
    indicator=True
)

print("===== KẾT QUẢ MERGE =====")
print("Kích thước:", df_final.shape)
display(df_final.head(10))

print("\nTình trạng ghép:")
display(df_final["_merge"].value_counts())


In [ ]:
# Kiểm tra các dòng chưa tìm thấy thông tin tĩnh
unmatched = (
    df_final.loc[df_final["_merge"] == "left_only", ["MovieName"]]
    .drop_duplicates()
)

print("Số MovieName chưa ghép được:", len(unmatched))
display(unmatched.head(20))

# Kiểm tra missing ở các cột thông tin tĩnh
static_info_cols = [
    c for c in [
        "MovieURL", "Rating", "Director", "Actors",
        "Genre", "ReleaseDate", "Duration", "Language"
    ] if c in df_final.columns
]

print("Missing sau merge:")
display(df_final[static_info_cols].isna().sum().to_frame("Missing"))


## 7. Lưu kết quả

- `cgv_static_master.csv`: Bảng Dữ liệu Tĩnh Hợp nhất sau `concat`.
- `cgv_final_integrated.csv`: Bảng cuối sau `merge`, gồm dữ liệu lịch chiếu + AI + thông tin phim.


In [ ]:
static_output = DATA_DIR / "cgv_static_master.csv"
final_output = DATA_DIR / "cgv_final_integrated.csv"

df_static_master.to_csv(
    static_output,
    index=False,
    encoding="utf-8-sig"
)

df_final_export = df_final.drop(columns=["_merge"])

df_final_export.to_csv(
    final_output,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:")
print("-", static_output)
print("-", final_output)


# KẾT LUẬN

**Nối dọc (`CONCAT`):**

`cgv_movie_details.csv` + `cgv_upcoming_movie(s).csv`
→ kiểm tra → chuẩn hóa → `pd.concat()`
→ **Bảng Dữ liệu Tĩnh Hợp nhất**

**Gộp ngang (`MERGE`):**

`cgv_showtimes_AI_generated.csv` + Bảng Dữ liệu Tĩnh Hợp nhất
→ khóa chung `MovieName`
→ **LEFT JOIN**
→ **Bảng dữ liệu tích hợp cuối cùng**

`LEFT JOIN` được chọn vì cần bảo toàn toàn bộ các suất chiếu của bảng động, sau đó mới bổ sung thông tin phim từ bảng tĩnh.
